In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader,TensorDataset,random_split
from PIL import Image
import PIL
import numpy as np
import pandas as pd
import random
import torch.nn.functional as F
import glob
import cv2 as cv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split
from tqdm import tqdm
#from torchinfo import summary
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import torchmetrics as tm
import seaborn as sns

import time
import splitfolders
import warnings
warnings.simplefilter('ignore')

from typing import Callable, Optional
from itertools import product

import segmentation_models_pytorch as smp
from torchvision.transforms import v2
from torchvision.io import read_image
from torchvision import tv_tensors
# from torchmetrics.functional.classification import dice
# from torchmetrics.classification import Dice
# from torchmetrics import Dice
from torchmetrics.aggregation import MeanMetric
from torchmetrics import JaccardIndex, Precision, Recall, F1Score
from torchmetrics.detection import IntersectionOverUnion
from torchmetrics.classification import Dice


import sys

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
def num_trainable_parameters(model):
    nums = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    return nums

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# image = Image.open(r'D:\University\Master\Thesis\naturally fractured coal reservoir ((Dataset))\Naturally Fractured coal Rocks\new_labeled2\New folder\intact-coal (1245)_128_256.tif')
image = Image.open(r'C:\1 image_label\intact-coal (2920)_512_512.tif')
plt.imshow(image, cmap = 'gray')
array = np.array(image)
tens = torch.tensor(array, dtype = torch.int64)
tens.unique()

# Unet

In [ ]:
real_images_folder = 
label_images_folder = 

# seed = 8
batch_size = 32
backbone = 'efficientnet-b3'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lr = 0.1
momentum = 0.9
wd = 1e-4
num_epochs = 20

In [ ]:
# transform_real = transforms.Compose([
#     #transforms.RandomHorizontalFlip(p = 0.5),
#     #transforms.RandomVerticalFlip(p = 0.5),
#     #transforms.Grayscale(num_output_channels = 1),
#     #transforms.Resize((128,128)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean = [0.3362], std = [0.0681])]
#                             )

# transform_label = transforms.Compose([
#     #transforms.RandomHorizontalFlip(p = 0.5),
#     #transforms.RandomVerticalFlip(p = 0.5),
#     #transforms.Grayscale(num_output_channels = 1),
#     #transforms.Resize((128,128)),
#     transforms.ToTensor()]
#                             )

transform_train_real = v2.Compose([
#    v2.Resize(size=(234,), antialias=True),
#    v2.RandomCrop(size=(100, 100)),
#    v2.RandomPhotometricDistort(p=0.5),
#    v2.RandomHorizontalFlip(p=0.5),
    #v2.ToTensor(),
    v2.Lambda(lambda x: (x - x.min()) / (x.max() - x.min())),
    v2.Normalize(mean=(0.5,), std=(0.5,)),
    ])

transform_eval_real = v2.Compose([
    #v2.Resize(size=(224, 224), antialias=True),
    #v2.ToTensor(),
    v2.Lambda(lambda x: (x - x.min()) / (x.max() - x.min())),
    v2.Normalize(mean=(0.5,), std=(0.5,))
    ])

transform_train_label = v2.Compose([
#    v2.Resize(size=(234,), antialias=True),
#    v2.RandomCrop(size=(100, 100)),
#    v2.RandomPhotometricDistort(p=0.5),
#    v2.RandomHorizontalFlip(p=0.5),
    #v2.ToTensor(),
    #v2.Lambda(lambda x: (x - x.min()) / (x.max() - x.min())),
    v2.Normalize(mean=(0.5,), std=(0.5,))
    ])

transform_eval_label = v2.Compose([
    #v2.Resize(size=(224, 224), antialias=True),
    #v2.ToTensor(),
    #v2.Lambda(lambda x: (x - x.min()) / (x.max() - x.min())),
    v2.Normalize(mean=(0.5,), std=(0.5,))
    ])

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, real_image_folder, label_image_folder, real_images, label_images,real_transform: Optional[Callable] = None, label_transform: Optional[Callable] = None):
        self.real_image_paths = real_image_folder
        self.label_image_paths = label_image_folder
        self.real_images = real_images
        self.label_images = label_images
        self.real_transform = real_transform
        self.label_transform = label_transform

    def __len__(self):
        return len(self.real_image_paths)

    def __getitem__(self, idx):
        
        real_img_path = os.path.join(self.real_image_paths, self.real_images[idx])
        label_img_path = os.path.join(self.label_image_paths, self.label_images[idx])
        # Open the images correctly using the full paths
        real_img = Image.open(real_img_path)#.convert('L')  # 8-bit grayscale ('L')
        label_img = Image.open(label_img_path)#.convert('L')  # 8-bit grayscale ('L')

        real_img = torch.tensor(np.array(real_img), dtype = torch.float32)  # Convert to tensor
        # Apply transformations for real and label images if provided
        
        if real_img.ndimension() == 2:  
            real_img = real_img.unsqueeze(0)  # Add a channel dimension
            
        if self.real_transform:
            real_img = self.real_transform(real_img)

        label_img = torch.tensor(np.array(label_img), dtype = torch.int64)

        if label_img.ndimension() == 2:  
            label_img = label_img.unsqueeze(0)  # Add a channel dimension

        if self.label_transform:
            label_img = self.label_transform(label_img)

        return real_img, label_img

In [ ]:
real_images_folder = real_images_folder
label_images_folder = label_images_folder
real_images = os.listdir(real_images_folder)
label_images = os.listdir(label_images_folder)

In [ ]:
len(real_images)

In [ ]:
len(label_images)

In [ ]:
real_images

In [ ]:
label_images

In [ ]:
x_train_real, x_test_real, y_train_label, y_test_label = train_test_split(real_images, label_images, test_size=0.8, random_state=42)
x_test, x_valid, y_test, y_valid = train_test_split(x_test_real, y_test_label, test_size=0.2, random_state=42)

In [ ]:
def tile(filename, dir_in, dir_out, d):
    """
    Crops an image into dxd tiles and saves them.
    Keeps the same filename prefix with indices.
    """
    name, ext = os.path.splitext(filename)
    img_path = os.path.join(dir_in, filename)

    if not os.path.exists(img_path):
        print(f"Skipping missing file: {img_path}")
        return

    img = Image.open(img_path)
    w, h = img.size
    grid = product(range(0, h, d), range(0, d, d))

    for i, j in grid:
        box = (j, i, d, i)
        tile_img = img.crop(box).resize((d, d), Image.LANCZOS)
        out_path = os.path.join(dir_out, f"{name}_{i}_{j}{ext}")
        tile_img.save(out_path)

#process dataset splits 
def crop_split(split_name, real_files, label_files, real_dir, label_dir, out_base, tile_size=128):
    real_out = os.path.join(out_base, split_name, "real")
    label_out = os.path.join(out_base, split_name, "label")
    os.makedirs(real_out, exist_ok=True)
    os.makedirs(label_out, exist_ok=True)

    for real, label in zip(real_files, label_files):
        tile(real, real_dir, real_out, tile_size)
        tile(label, label_dir, label_out, tile_size)

    print(f"{split_name} set cropped and saved to {os.path.join(out_base, split_name)}")

#Run cropping 
crop_split("train", x_train_real, y_train_label, real_images_folder, label_images_folder, output_base_dir)
crop_split("val", x_val_real, y_val_label, real_images_folder, label_images_folder, output_base_dir)
crop_split("test", x_test_real, y_test_label, real_images_folder, label_images_folder, output_base_dir)

print("Done cropping all splits!")

In [ ]:
print(len(x_train_real), len(y_train_label))  
print(len(x_valid), len(y_valid))  
print(len(x_test), len(y_test)) 

In [ ]:
train_dataset = SegmentationDataset(real_images_folder, label_images_folder, x_train_real, y_train_label, real_transform = transform_train_real, label_transform = transform_train_label)
valid_dataset = SegmentationDataset(real_images_folder, label_images_folder, x_valid, y_valid, real_transform = transform_eval_real, label_transform = transform_eval_label)
test_dataset = SegmentationDataset(real_images_folder, label_images_folder, x_test, y_test, real_transform = transform_eval_real, label_transform = transform_eval_label)

len(train_dataset), len(valid_dataset), len(test_dataset)

In [ ]:
test_dataset[0]

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


print(f"Number of training batches: {len(train_loader)}")
print(f"Number of valid batches: {len(valid_loader)}")
print(f"Number of test batches: {len(test_loader)}")

In [ ]:
imgs, msks = next(iter(train_loader))
imgs.shape, msks.shape

In [ ]:
imgs, msks = next(iter(valid_loader))
imgs.shape, msks.shape

In [ ]:
imgs, msks = next(iter(test_loader))
imgs.shape, msks.shape

In [ ]:
imgs, msks = next(iter(train_loader))
imgs.shape, msks.shape

In [ ]:
for inputs, labels in train_loader:
    print(inputs)
    print(labels)
    print(f'Inputs type: {type(inputs)}')
    print(f'Labels type: {type(labels)}')
    print(f'shape of the inputs are {inputs.shape}')
    print(f'shape of the labels are {labels.shape}')
    print('\n')

In [ ]:
for inputs, labels in valid_loader:
    print(inputs)
    print(labels)
    print(f'Inputs type: {type(inputs)}')
    print(f'Labels type: {type(labels)}')
    print(f'shape of the inputs are {inputs.shape}')
    print(f'shape of the labels are {labels.shape}')
    print('\n')

In [ ]:
# plt.figure(figsize=(15,15))
# plt.subplot(1,2,1)
# plt.imshow(inputs[3].squeeze(0).cpu().numpy(), cmap='gray')  
# plt.subplot(1,2,2)
# plt.imshow(labels[3].squeeze(0).cpu().numpy(), cmap='gray')  # Convert to numpy for displaying
# plt.title("Label Image")
# plt.show()

## Modeling

In [ ]:
unet = smp.Unet(encoder_name = backbone)

In [ ]:
imgs, masks = next(iter(train_loader))
unet(imgs).shape

In [ ]:
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing= True)

In [ ]:
unet = unet.to(device)
start.record()

with torch.no_grad():
    out = unet(imgs.to(device))
end.record()

torch.cuda.synchronize()
start.elapsed_time(end)

#ms - milli seconds

In [ ]:
loss_fn = smp.losses.DiceLoss(mode = 'multilabel')

In [ ]:
# MeanMetric is like AverageMeter class.


metric = Dice(num_classes=3, multiclass=True).to(device)

In [ ]:
# def train_one_epoch(model, train_loader, loss_fn, optimizer, metric, epoch=None):
#   model.train()
#   loss_train = MeanMetric()
#   metric.reset()
#   with tqdm(train_loader, unit="batch") as tepoch:
#     for inputs, targets in tepoch:
#       if epoch is not None:
#         tepoch.set_description(f"Epoch {epoch}")
#       inputs = inputs.to(device)
#       targets = targets.to(device)

#       outputs = model(inputs)
#       # print(outputs.shape)
#       # print(outputs.dtype)
#       # print(outputs.min())
#       # print(outputs.max())
#       outputs = torch.softmax(outputs, dim = 1)
#       loss = loss_fn(outputs, targets)

#       loss.backward()
#       optimizer.step()
#       optimizer.zero_grad()

#       loss_train.update(loss.item(), weight = len(targets))
#       metric.update(outputs, targets)
#       tepoch.set_postfix(loss=loss_train.compute().item(), 
#                          metric=metric.compute().item())
#   return model, loss_train.compute().item(), metric.compute().item()



def train_one_epoch(model, train_loader, loss_fn, optimizer,epoch=None): #metric, 
    model = model.to(device)
    model.train()
    loss_train = MeanMetric()  # Tracks the training loss
    #metric.reset()  # Reset the metric at the beginning of each epoch
    
    with tqdm(train_loader, unit="batch") as tepoch:
        for inputs, targets in tepoch:
            if epoch is not None:
                tepoch.set_description(f"Epoch {epoch}")
            
            # Move the data to the correct device (GPU/CPU)
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Forward pass: Get model outputs
            outputs = model(inputs)


            # print(f"Model device: {model.device}")
            # print(f"Input device: {inputs.device}")
            # print(f"Target device: {targets.device}")
            # print(f"Outputs shape: {outputs.shape}")
            
            # Apply softmax to the outputs to get class probabilities (multi-class)
            #outputs = torch.softmax(outputs, dim=1)  # Softmax over the class dimension
            
            # Compute the loss
            loss = loss_fn(outputs, targets)
            
            # Backward pass: Compute gradients
            loss.backward()

            # Update training loss metric (MeanMetric) with the current loss
            loss_train.update(loss.item(), weight=len(targets))
            
            # Update Dice metric for multi-class segmentation
            #metric.update(outputs, targets)

            # Set the progress bar description with current loss and metric values
            tepoch.set_postfix(loss=loss_train.compute().item(), )
                              # metric=metric.compute().item())

    # Return the model, final loss, and metric (Dice score)
    return model, loss_train.compute().item()#, metric.compute().item()


In [ ]:
# def validation(model, valid_loader, loss_fn, metric):
#   model.eval()
#   loss_valid = MeanMetric()
#   metric.reset()
#   with torch.inference_mode():
#     for i, (inputs, targets) in enumerate(valid_loader):
#       inputs = inputs.to(device)
#       targets = targets.to(device)

#       outputs = model(inputs)
#       outputs = torch.softmax(outputs, dim = 1)
        
#       loss = loss_fn(outputs, targets)
    
#       loss_valid.update(loss.item(), weights = len(targets))
#       metric(outputs, targets)
#   return loss_valid.compute().item(), metric.compute().item()


import torch
from torchmetrics.classification import Dice
from torchmetrics import MeanMetric

def validation(model, valid_loader, loss_fn):#metric, device):
    model = model.to(device)
    model.eval()  # Set model to evaluation mode
    loss_valid = MeanMetric()  # Tracks the validation loss
    #metric.reset()  # Reset the metric at the beginning of the validation phase
    
    # Disable gradient computation (for faster inference)
        for i, (inputs, targets) in enumerate(valid_loader):
            inputs = inputs.to(device)  # Move data to the correct device
            targets = targets.to(device)  # Move targets to the correct device

            # Forward pass: Get model outputs
            outputs = model(inputs)
            
            # Apply softmax to the outputs to get class probabilities (multi-class)
            #outputs = torch.softmax(outputs, dim=1)  # Softmax over the class dimension
            
            # Compute the loss (e.g., cross-entropy loss)
            loss = loss_fn(outputs, targets)
            
            # Update the validation loss metric
            loss_valid.update(loss.item(), weight=len(targets))
            
            # Update the Dice metric for multi-class segmentation
            #metric.update(outputs, targets)

    # Return the computed validation loss and Dice score
    return loss_valid.compute().item()#, metric.compute().item()


In [ ]:
model = smp.Unet(encoder_name = backbone,
               classes = num_classes).to(device)
inputs, targets = next(iter(train_loader))
inputs = inputs.to(device)
targets = targets.to(device)

with torch.inference_mode():
    outputs = model(inputs)
    loss = loss_fn(outputs, targets)

print(loss)

In [ ]:
mini_valid_size = 500
# use valid set because our training model may has some augmentations.
_, mini_valid_dataset = random_split(valid_dataset, (len(valid_dataset)-mini_valid_size, mini_valid_size))
mini_valid_loader = DataLoader(mini_valid_dataset, 20, shuffle=True)

In [ ]:
model = smp.Unet(encoder_name=backbone).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = smp.losses.DiceLoss(mode='multilabel')

In [ ]:
num_epochs = 50
for epoch in range(num_epochs):
  model, _= train_one_epoch(model, mini_valid_loader, loss_fn, optimizer, epoch+1)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
num_epochs = 3

for lr in [0.9, 0.5, 0.3, 0.1, 0.01, 0.001]:
  print(f'LR={lr}')

  model = smp.Unet(encoder_name=backbone).to(device)
  optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

  for epoch in range(num_epochs):
    # model, _, _ = train_one_epoch(model, train_loader, loss_fn, optimizer, metric, epoch+1)
    model, _ = train_one_epoch(model, train_loader, loss_fn, optimizer, epoch+1)


  print()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
set_seed(seed)
model = model = smp.Unet(encoder_name=backbone).to(device)

In [ ]:
set_seed(seed)
lr = 0.001
wd = 1e-4
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
lr_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10, 15], gamma=0.1)
# we have steplr as well rather than MultiStepLR and steplr change the lr after passing specific numbers of epochs.

In [ ]:
import torch
import torch.optim as optim
import segmentation_models_pytorch as smp
from torchmetrics.classification import JaccardIndex, Precision, Recall, F1Score
from torchmetrics.detection import IntersectionOverUnion
from torchmetrics import Dice
import numpy as np
import random
import pandas as pd

# -----------------
# Utility: Set Seed
# -----------------
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------
# Metric Calculation
# -----------------
def compute_metrics(y_true, y_pred, num_classes, device):
    """Compute IoU, Precision, Recall, F1, Jaccard, Dice (all from torchmetrics)."""
    metrics = {}

    # IoU (global)
    iou_metric = IntersectionOverUnion(class_metrics=False).to(device)
    iou_val = iou_metric(
        [{"masks": (y_pred == c).to(torch.uint8).unsqueeze(0)} for c in range(num_classes)],
        [{"masks": (y_true == c).to(torch.uint8).unsqueeze(0)} for c in range(num_classes)]
    )
    metrics["IoU (%)"] = float(iou_val["iou"].mean().cpu()) * 100

    # Jaccard (macro per-class)
    jaccard_macro = JaccardIndex(task="multiclass", num_classes=num_classes, average="macro").to(device)
    metrics["Jaccard Score (%)"] = jaccard_macro(y_pred, y_true).item() * 100

    # Precision / Recall / F1 (macro)
    precision = Precision(task="multiclass", num_classes=num_classes, average="macro").to(device)
    recall = Recall(task="multiclass", num_classes=num_classes, average="macro").to(device)
    f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro").to(device)

    metrics["Precision (%)"] = precision(y_pred, y_true).item() * 100
    metrics["Recall (%)"] = recall(y_pred, y_true).item() * 100
    metrics["F1-score (%)"] = f1(y_pred, y_true).item() * 100

    # Dice (macro)
    dice_metric = Dice(num_classes=num_classes, average="macro").to(device)
    metrics["Dice Metric (%)"] = dice_metric(y_pred, y_true).item() * 100

    return metrics


# -----------------
# Training Loop (per seed)
# -----------------
def run_training_for_seed(seed, backbone, num_classes, device, train_loader, valid_loader, loss_fn, num_epochs=60):
    set_seed(seed)
    print(f"\n========== Training with seed {seed} ==========")

    model = model
    best_loss_valid = torch.inf
    loss_train_hist, loss_valid_hist = [], []

    for epoch in range(num_epochs):
        model.train()
        model, loss_train = train_one_epoch(model, train_loader, loss_fn, optimizer, epoch + 1)

        model.eval()
        loss_valid = validation(model, valid_loader, loss_fn)

        loss_train_hist.append(loss_train)
        loss_valid_hist.append(loss_valid)

        if loss_valid < best_loss_valid:
            torch.save(model.state_dict(), f"model_seed{seed}.pt")
            best_loss_valid = loss_valid
            print(f"Model saved (seed {seed})!")

        print(f"[Epoch {epoch+1}/{num_epochs}] Valid Loss: {loss_valid:.4f}, LR: {lr_scheduler.get_last_lr()[0]:.6f}")
        lr_scheduler.step()

    # Load best model for metric computation
    model.load_state_dict(torch.load(f"model_seed{seed}.pt"))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in valid_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            preds = torch.argmax(preds, dim=1)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_labels)

    metrics = compute_metrics(y_true, y_pred, num_classes, device)
    return metrics


# -----------------
# Main Experiment
# -----------------
seeds = [1, 3, 5, 7, 8]
all_results = []

for s in seeds:
    metrics = run_training_for_seed(
        seed=s,
        backbone=backbone,
        num_classes=num_classes,
        device=device,
        train_loader=train_loader,
        valid_loader=valid_loader,
        loss_fn=loss_fn,
        num_epochs=num_epochs
    )
    all_results.append((s, metrics))

# -----------------
# Summary Table
# -----------------
results_df = pd.DataFrame(
    [{**m, "Seed": s} for s, m in all_results]
).set_index("Seed")

print("\n===== Final Results Across Seeds =====")
print(results_df.round(3))
print("\nMean metrics:")
print(results_df.mean().round(3))


In [ ]:
set_seed(8)
model = model = smp.Unet(encoder_name=backbone).to(device)

In [ ]:
set_seed(8)
lr = 0.001
wd = 1e-4
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
lr_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10, 15], gamma=0.1)
# we have steplr as well rather than MultiStepLR and steplr change the lr after passing specific numbers of epochs.

In [ ]:
loss_train_hist = []
loss_valid_hist = []

metric_train_hist = []
metric_valid_hist = []

best_loss_valid = torch.inf
epoch_counter = 0

In [ ]:
def compute_metrics_torchmetrics(model, dataloader, num_classes, device="cpu"):
    model.eval()
    preds_all = []
    labels_all = []

    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            preds = model(imgs)
            preds = torch.argmax(preds, dim=1)
            preds_all.append(preds.cpu())
            labels_all.append(labels.cpu())

    preds_all = torch.cat(preds_all)
    labels_all = torch.cat(labels_all)
    preds_np = preds_all.numpy()
    labels_np = labels_all.numpy()
    labels = np.arange(num_classes)

    # IoU using detection metric (global)
    iou_metric = IntersectionOverUnion(class_metrics=False).to(device)
    preds_list = []
    targets_list = []
    for c in range(num_classes):
        pred_mask = (preds_all == c).to(torch.uint8).unsqueeze(0).to(device)
        gt_mask = (labels_all == c).to(torch.uint8).unsqueeze(0).to(device)
        preds_list.append({"masks": pred_mask})
        targets_list.append({"masks": gt_mask})
    iou_value = iou_metric(preds_list, targets_list)
    IoU_percent = float(iou_value["iou"].mean().cpu()) * 100

    # Dice using torchmetrics (macro)
    dice_metric = Dice(num_classes=num_classes, average="macro").to(device)
    Dice_percent = float(dice_metric(preds_all.to(device), labels_all.to(device)).cpu()) * 100

    # Jaccard, Precision, Recall, F1 (macro)
    jaccard_macro = jaccard_score(
        labels_np.flatten(), preds_np.flatten(), labels=labels, average="macro", zero_division=0
    ) * 100
    precision = precision_score(
        labels_np.flatten(), preds_np.flatten(), labels=labels, average="macro", zero_division=0
    ) * 100
    recall = recall_score(
        labels_np.flatten(), preds_np.flatten(), labels=labels, average="macro", zero_division=0
    ) * 100
    f1 = f1_score(
        labels_np.flatten(), preds_np.flatten(), labels=labels, average="macro", zero_division=0
    ) * 100

    return {
        "IoU (%)": IoU_percent,
        "Jaccard Score (%)": jaccard_macro,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "F1-score (%)": f1,
        "Dice Metric (%)": Dice_percent
    }

#  TRAINING LOOP WITH METRICS

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    model, loss_train = train_one_epoch(model, train_loader, loss_fn, optimizer, epoch + 1)

    model.eval()
    loss_valid = validation(model, valid_loader, loss_fn)

    # Compute metrics on validation set
    metrics = compute_metrics_torchmetrics(model, valid_loader, num_classes, device=device)

    loss_train_hist.append(loss_train)
    loss_valid_hist.append(loss_valid)

    if loss_valid < best_loss_valid:
        torch.save(model, f"model.pt")
        best_loss_valid = loss_valid
        print("Model Saved!")

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {loss_train:.4f} | Valid Loss: {loss_valid:.4f} | LR: {lr_scheduler.get_last_lr()[0]:.6f}")
    print(
        f"IoU: {metrics['IoU (%)']:.2f} | Jaccard: {metrics['Jaccard Score (%)']:.2f} | "
        f"F1: {metrics['F1-score (%)']:.2f} | Precision: {metrics['Precision (%)']:.2f} | "
        f"Recall: {metrics['Recall (%)']:.2f} | Dice: {metrics['Dice Metric (%)']:.2f}"
    )
    print()

    lr_scheduler.step()
    epoch_counter += 1


In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(range(epoch_counter), loss_train_hist, 'r-', label='Train')
plt.plot(range(epoch_counter), loss_valid_hist, 'b-', label='Validation')

plt.xlabel('Epoch')
plt.ylabel('loss')
plt.grid(True)
plt.legend()

In [ ]:
model_path = 'model.pt'
model = torch.load(model_path, weights_only = False)
model.eval()

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
_, metric_test = validation(model, test_loader, loss_fn, metric)
metric_test

In [ ]:
model_path = 'model.pt'
model = torch.load(model_path, weights_only = False)
model.eval()

In [ ]:
def segment(image, model):
  with torch.inference_mode():
    prediction = model(image)
    return torch.sigmoid(prediction).cpu()
      #the amounts must be between 0 and 1 so we use sigmoid. our mask is between 0 and 1 as well.

In [ ]:
model_path = 'model.pt'
model = torch.load(model_path)
model.eval()

In [ ]:
outputs = []

for img, mask in new_test_loader:
  output = segment(img.to(device), model)
  outputs.append(output)

In [ ]:
large_bowel_dice, small_bowel_dice, stomach_dice = Dice(), Dice(), Dice()

for output, (_, mask) in zip(outputs, new_test_loader):
  large_bowel_dice.update(output[0, 0], mask[0, 0])
  small_bowel_dice.update(output[0, 1], mask[0, 1])
  stomach_dice.update(output[0, 2], mask[0, 2])

print(f'Large Bowel Dice: {large_bowel_dice.compute().item()}')
print(f'Small Bowel Dice: {small_bowel_dice.compute().item()}')
print(f'Stomach Dice: {stomach_dice.compute().item()}')

In [ ]:
tp, fp, fn, tn = smp.metrics.get_stats(outputs[-1], mask, mode='multilabel', threshold=0.5)
tp, fp, fn, tn

In [ ]:
imgs, masks = next(iter(new_test_loader))

outputs = segment(imgs.to(device), model)

tp, fp, fn, tn = smp.metrics.get_stats(outputs[[-1]], masks[[-1]], mode='multilabel', threshold=0.5)
tp#, fp, fn, tn

In [ ]:
outputs[0].shape

In [ ]:
smp.metrics.accuracy(tp, fp, fn, tn)

In [ ]:
stat_scores = tm.StatScores(task='multilabel', num_labels=3, average=None)
stat_scores.update(outputs[[-1]], masks[[-1]])
stat_scores.compute()